# Chapter 5 · Retrieval-augmented generation (RAG)

The first assistant that answers with the North Cycles documents: retrieve the chunks most similar to the question, build the prompt and generate the answer with citations. We test it with twenty questions, fifteen answered in the documents and five not, and we improve it with worked examples and with a check before answering.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*. The notebook loads two models: the language model (Qwen2.5-1.5B-Instruct with a GPU, Qwen3-0.6B without one) and the embedding model of chapter 4.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. The models and the knowledge base

In [ ]:
"""Chapter 5: Retrieval-augmented generation (RAG). On GPU or CPU."""
import json
import re
import time
import urllib.request
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_GPU = "Qwen/Qwen2.5-1.5B-Instruct"  # with Colab's free T4 GPU
MODEL_CPU = "Qwen/Qwen3-0.6B"             # without a GPU: smaller and faster
MODEL = None                              # None: choose according to the setup
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
DATA = Path("data")
REPOSITORY = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/en/"



def load(name=None):
    """The language model and the embedding model, on the GPU if there is one."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if name is None:
        name = MODEL_GPU if device == "cuda" else MODEL_CPU
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
    model.to(device).eval()
    embeddings = SentenceTransformer(EMBEDDING_MODEL, device=device)
    print(f"Models {name} and {EMBEDDING_MODEL} on {device}")
    return tok, model, embeddings


def download(name):
    """A file from the data folder; in Colab, it is downloaded from the repository."""
    path = DATA / name
    if not path.exists():
        DATA.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORY + "data/" + name, path)
    return path.read_text(encoding="utf-8")


def knowledge_base():
    """The chunks we search: the FAQ and the sections of the terms and
    conditions, each with its source."""
    faq = json.loads(download("faq.json"))["questions"]
    chunks = [{"source": f"FAQ, {q['id']}",
               "text": q["question"] + " " + q["answer"]} for q in faq]
    terms = download("terms_and_conditions.md")
    for section in re.split(r"\n(?=## )", terms)[1:]:
        title, text = section.strip().split("\n", 1)
        chunks.append({"source": "Terms and conditions, " + title[3:],
                       "text": text.strip()})
    return chunks

In [ ]:
tok, model, embeddings = load(MODEL)
chunks = knowledge_base()
print(len(chunks), "chunks. Two of them:")
for c in (chunks[16], chunks[36]):
    print(f"- ({c['source']}) {c['text'][:150]}...")

## 2. Retrieving

In [ ]:
class Searcher:
    """The semantic search engine of chapter 4."""

    def __init__(self, chunks):
        """Computes the embedding of every chunk."""
        self.chunks = chunks
        self.matrix = embeddings.encode(
            ["passage: " + c["text"] for c in chunks], normalize_embeddings=True)

    def search(self, query, k=3):
        """The k chunks most similar to the query."""
        vector = embeddings.encode(["query: " + query], normalize_embeddings=True)
        scores = self.matrix @ vector[0]
        return [self.chunks[i] for i in np.argsort(-scores)[:k]]

In [ ]:
searcher = Searcher(chunks)
for c in searcher.search("Can I pay for a $900 bike over 12 months?"):
    print(c["source"])

## 3. Building the prompt and generating

In [ ]:
SYSTEM_NO_DOCUMENTS = ("You are the customer service assistant of North Cycles, "
                       "a bicycle store in Portland, Oregon. Answer in English, "
                       "in two or three sentences.")


DONT_KNOW = "I don't have that information; I'm passing your question to the team."


SYSTEM_RAG = (
    "You are the customer service assistant of North Cycles, a bicycle store in "
    "Portland, Oregon. Answer the customer's question in English, in two or three "
    "sentences, using only the information in the documents you are given. After "
    "each fact, give the number of the document it comes from in square brackets, "
    "for example [2]. If the documents do not contain the answer, reply exactly: "
    f"\"{DONT_KNOW}\" Do not make anything up.")


def rag_prompt(question, documents):
    """The user message: the numbered documents and the question."""
    listing = "\n\n".join(f"[{i}] ({d['source']}) {d['text']}"
                          for i, d in enumerate(documents, 1))
    return f"Documents:\n\n{listing}\n\nCustomer's question: {question}"

In [ ]:
question = "Can I pay for a $900 bike over 12 months?"
print(rag_prompt(question, searcher.search(question)))

In [ ]:
def ask(system, user, examples=(), max_tokens=200):
    """Sends a conversation to the model and returns its answer as text."""
    messages = [{"role": "system", "content": system}]
    for question, answer in examples:
        messages += [{"role": "user", "content": question},
                     {"role": "assistant", "content": answer}]
    messages.append({"role": "user", "content": user})
    text = tok.apply_chat_template(messages, tokenize=False,
                                   add_generation_prompt=True,
                                   enable_thinking=False)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, pad_token_id=tok.eos_token_id)
    new = output[0, inputs["input_ids"].shape[1]:]
    reply = tok.decode(new, skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", reply, flags=re.S).strip()


def respond(question, searcher, k=3):
    """RAG: retrieve k chunks, build the prompt and generate."""
    if k == 0:
        return ask(SYSTEM_NO_DOCUMENTS, question), []
    documents = searcher.search(question, k)
    return ask(SYSTEM_RAG, rag_prompt(question, documents)), documents

In [ ]:
for question in ["Can I pay for a $900 bike over 12 months?",
                 "How much does the City 5 bike weigh?"]:
    print("Question:", question)
    print("  No documents:", respond(question, searcher, k=0)[0])
    print("  With RAG:", respond(question, searcher)[0], "\n")

## 4. Two improvements: worked examples and a check first

In [ ]:
RAG_EXAMPLES = [
    (rag_prompt("Do you have helmets for my five-year-old son?", [
        {"source": "Catalog, helmets", "text": "Kids' helmets in sizes XS "
         "(18-19.5 in) and S (19.5-21 in), recommended for ages 3 to 8."},
        {"source": "Terms and conditions, 3. Prices",
         "text": "Prices are in US dollars; North Cycles does not charge sales tax."}]),
     "Yes, we have kids' helmets in sizes XS and S, recommended for ages 3 to 8 "
     "[1]."),
    (rag_prompt("Do you sell electric scooters?", [
        {"source": "Catalog, e-bikes", "text": "City and mountain e-bikes, "
         "with a range of 40 to 75 miles."},
        {"source": "FAQ, 99", "text": "Do you rent bikes? "
         "Yes, city bikes and e-bikes by the day."}]),
     DONT_KNOW),
]


SYSTEM_CHECK = ("Decide whether the documents contain the information needed to "
                "answer the customer's question. Answer only \"yes\" or \"no\".")


def respond_with_examples(question, searcher, k=3):
    """RAG with two worked examples: one that cites and one that doesn't know."""
    documents = searcher.search(question, k)
    prompt = rag_prompt(question, documents)
    return ask(SYSTEM_RAG, prompt, RAG_EXAMPLES), documents


def respond_checking(question, searcher, k=3):
    """Before answering, asks the model whether the documents are enough."""
    documents = searcher.search(question, k)
    prompt = rag_prompt(question, documents)
    verdict = ask(SYSTEM_CHECK, prompt, max_tokens=3)
    if verdict.lower().startswith("no"):
        return DONT_KNOW, documents
    return ask(SYSTEM_RAG, prompt, RAG_EXAMPLES), documents

## 5. Evaluating with twenty questions

In [ ]:
# Question and texts that a correct answer must contain (one is enough);
# None if the answer is not in the documents.
QUESTIONS = [
    ("Can I return bib shorts if I've already opened them?",
     ["hygiene", "not accepted", "aren't accepted"]),
    ("How much does it cost to ship a $30 order to Chicago?", ["4.95"]),
    ("How many days do I have to return a bike?", ["30 days", "30 calendar days"]),
    ("If I pay by bank transfer, when do you prepare the order?", ["receiv"]),
    ("What warranty does an e-bike battery have?",
     ["2 years", "two years", "2-year", "two-year"]),
    ("How much is a tune-up at your repair shop?", ["$35", "35 dollars"]),
    ("Can I pay for a $900 bike over 12 months?",
     ["interest-free", "interest free", "without interest", "no interest"]),
    ("What do I do if the box arrives dented?", ["delivery receipt"]),
    ("How long do you take to answer an email?", ["48 hours", "48 business hours"]),
    ("Do the bikes arrive assembled?", ["90"]),
    ("What range does an e-bike have?", ["75 miles"]),
    ("What time does the store open on Saturdays?", ["10 a.m.", "10 am", "10:00"]),
    ("What's the most expensive gift card you have?", ["$100", "100 dollars"]),
    ("If I return only part of an order, do you refund the shipping?",
     ["whole order", "entire order", "complete order", "full order"]),
    ("Can I cancel an order that has already left the warehouse?",
     ["has not left", "hasn't left", "has not yet left", "before it leaves"]),
    ("How much does the City 5 bike weigh?", None),
    ("Do you have customer parking at the store?", None),
    ("Do you organize group rides on Sundays?", None),
    ("Do you sell Shimano spare parts?", None),
    ("Do you offer student discounts?", None),
]

In [ ]:
def contains(text, keys):
    """Does the text contain any of the keys? Curly apostrophes count as straight."""
    text = text.lower().replace("’", "'")
    return any(k.lower() in text for k in keys)


def evaluate(searcher, variant):
    """Answers the 20 questions and records what goes right and what goes wrong."""
    rows = []
    for question, keys in QUESTIONS:
        start = time.time()
        answer, documents = variant(question, searcher)
        abstains = contains(answer, [DONT_KNOW[:30]])
        row = {"question": question, "answer": answer,
               "sources": [d["source"] for d in documents],
               "seconds": time.time() - start, "abstains": abstains}
        if keys is None:                        # not in the documents
            row["correct"] = abstains
        else:
            row["correct"] = contains(answer, keys) and not abstains
            row["retrieved"] = any(contains(d["text"], keys) for d in documents)
            row["cites"] = bool(re.search(r"\[\d\]", answer))
        rows.append(row)
    return rows


def summary(rows):
    """Correct answers, correct abstentions and retrieval failures."""
    answerable = [r for r in rows if "retrieved" in r]
    unanswerable = [r for r in rows if "retrieved" not in r]
    return {"correct": sum(r["correct"] for r in answerable),
            "retrieved": sum(r["retrieved"] for r in answerable),
            "wrong_abstentions": sum(r["abstains"] for r in answerable),
            "with_citations": sum(r["cites"] for r in answerable),
            "correct_abstentions": sum(r["correct"] for r in unanswerable),
            "seconds": round(sum(r["seconds"] for r in rows) / len(rows), 1)}

In [ ]:
VARIANTS = {
    "No documents": lambda q, s: respond(q, s, k=0),
    "RAG, 1 chunk": lambda q, s: respond(q, s, k=1),
    "RAG, 3 chunks": lambda q, s: respond(q, s, k=3),
    "RAG, 5 chunks": lambda q, s: respond(q, s, k=5),
    "With examples": respond_with_examples,
    "Checking first": respond_checking,
}

In [ ]:
results = {}
for name, variant in VARIANTS.items():
    results[name] = evaluate(searcher, variant)
    print(f"{name}:", summary(results[name]))

Read the answers of one variant, with a mark on the wrong ones:

In [ ]:
variant = "With examples"
for row in results[variant]:
    mark = "  " if row["correct"] else "✗ "
    print(mark + row["question"])
    print("     ", row["answer"][:200])

## Exercises

1. Explain in your own words the three phases of a RAG system. Which part of the system is responsible for each of the three kinds of failure in the chapter?

2. For each of the fifteen answerable questions, look at which chunks are retrieved with k = 3. For which one does retrieval fail? Why do you think the search engine does not find the right chunk?

3. Write five more questions whose answer is not in the documents and evaluate the variants with examples and with a check. Do the chapter's figures hold?

4. Replace the fixed phrase with a more natural one, such as "Sorry, I don't know." Does the number of times the model abstains change?

5. In the variant with a check, change the system message of the check so that it is less strict, for example: "Answer no only if no document deals with the topic of the question." Check that you get a result similar to the chapter's. Then try two other wordings. Which one best balances making things up and staying silent?

6. Write a function that checks the citations: that each number in square brackets corresponds to a retrieved document and that this document contains one of the question's keys. How many citations are correct?

7. Add a false chunk to the knowledge base: "Shipping to Hawaii is free on all orders." Ask how much it costs to ship to Hawaii. What does the assistant answer? What does this tell you about the quality of the documents?

8. Measure how many tokens the prompt has with 1, 3 and 5 chunks. How much does it grow? What would happen with 320-word chunks?

9. Change the order of the chunks so that the one most similar to the question goes last, next to the question, instead of first. Does the number of correct answers change?

10. **Question assistant.** Write a function `assistant(question)` that uses the variant you think is best and shows the answer followed by the cited sources. Try it with ten questions of your own and write a short report on what works and what doesn't.

## References

Models [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) and [multilingual-e5-base](https://huggingface.co/intfloat/multilingual-e5-base). RAG: [Lewis et al. (2020)](https://arxiv.org/abs/2005.11401). The North Cycles store and its documents are fictional. Code and experiments are our own.